# 12 — Prepare real MILK10k clinical images
Run on **CPU**, after notebook 11. Download the official 344.7 MB bundle, verify its pinned SHA-256 and ZIP CRCs, screen both clinical and paired dermoscopic images against all original split images and the Fitzpatrick training images used in notebook 09, then create prospective MILK-only train/validation/test manifests. No model training, test predictions, original split edits or app replacement.

MILK study team, DOI https://doi.org/10.34970/648456, CC-BY-NC. Image source: https://api.isic-archive.com/doi/milk10k/. Preserve attribution. MILK 0 (very dark) to 5 (very light) is **not Fitzpatrick**.

The pinned archive checksum was measured from the official download when this notebook was prepared; it is not a publisher-signed checksum. A future mismatch must be investigated, not bypassed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlopen
from PIL import Image
import io, json, hashlib, zipfile
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
# This is the completed notebook 11 audit you shared. Change only if rerunning a different audit.
AUDIT_ROOT = ROOT / 'Reports' / 'MILK10k_Audit' / '20261104_052201_662292'
EXTERNAL_MANIFEST = ROOT / 'Reports' / 'External_Data_Preparation' / 'external_training_manifest.csv'
SEED = 42
STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
DATA_ROOT = ROOT / 'Datasets' / 'MILK10k'
OUTPUT_ROOT = ROOT / 'Reports' / 'MILK10k_Preparation' / STAMP
SPLIT_ROOT = DATA_ROOT / 'Prepared' / STAMP
IMAGE_ROOT = DATA_ROOT / 'clinical_images'
for p in [OUTPUT_ROOT, SPLIT_ROOT, IMAGE_ROOT]: p.mkdir(parents=True, exist_ok=True)
CLASSES = ['BCC', 'Melanoma', 'SCC']
ARCHIVE_URL = 'https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/milk10k.zip'
ARCHIVE_SHA256 = '47cfe0870997657429d3578e40d22900521a7e0194b1e03fa946bfc4a3e93b58'
ARCHIVE = DATA_ROOT / 'milk10k.zip'

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()

candidate_path = AUDIT_ROOT / 'candidates_PENDING_image_checks.csv'
if not candidate_path.is_file():
    raise FileNotFoundError(f'Locate your completed notebook 11 report and update AUDIT_ROOT: {candidate_path}')
audit = json.loads((AUDIT_ROOT / 'audit_provenance.json').read_text())
if audit['source_doi'] != '10.34970/648456': raise ValueError('Wrong audit source.')
candidates = pd.read_csv(candidate_path, dtype={'isic_id': str, 'lesion_id': str})
if candidates.empty or candidates.isic_id.duplicated().any() or candidates.lesion_id.duplicated().any():
    raise ValueError('Empty or duplicate candidate identifiers.')
expected = {'Basal cell carcinoma':'BCC', 'Melanoma Invasive':'Melanoma',
            'Melanoma in situ':'Melanoma', 'Squamous cell carcinoma, Invasive':'SCC'}
if not candidates.diagnosis_full.map(expected).eq(candidates.class_label).all(): raise ValueError('Diagnosis mapping changed.')
if not candidates.diagnosis_confirm_type.eq('histopathology').all(): raise ValueError('Unconfirmed candidate diagnosis.')
if not candidates.image_type.eq('clinical: close-up').all(): raise ValueError('Unexpected image modality.')
reference_frames=[]
for split in ['train','validation','test']:
    p = ROOT / 'Datasets' / 'Splits' / f'{split}_metadata.csv'
    if sha256_file(p) != audit['original_split_sha256'][split]: raise ValueError('Original split changed since audit: '+split)
    f = pd.read_csv(p)
    f['reference_source'] = 'original_' + split
    reference_frames.append(f)
if not EXTERNAL_MANIFEST.is_file(): raise FileNotFoundError('Notebook 09 external training manifest required: '+str(EXTERNAL_MANIFEST))
f = pd.read_csv(EXTERNAL_MANIFEST)
if f.empty or not f['split'].eq('train').all(): raise ValueError('Expected nonempty external TRAINING manifest.')
f['reference_source'] = 'Fitzpatrick17k_training'
reference_frames.append(f)
references = pd.concat(reference_frames, ignore_index=True)
if references.image_path.isna().any(): raise ValueError('Missing reference image paths.')
print('Clinical candidates:',len(candidates))
print('Original and external reference image rows:',len(references))

## Download and verify
A completed ZIP is reused only if its checksum matches. Interrupted downloads leave a `.part` file; rerunning restarts that partial download. Source archives and existing images are not deleted or overwritten.

In [ ]:
if ARCHIVE.is_file():
    if sha256_file(ARCHIVE) != ARCHIVE_SHA256: raise ValueError('Cached archive differs; preserve and inspect it before retrying.')
    print('Using checksum-verified cached archive.')
else:
    partial = ARCHIVE.with_suffix('.zip.part')
    with urlopen(ARCHIVE_URL, timeout=120) as response, partial.open('wb') as out:
        written=0; last=0
        while True:
            block=response.read(4*1024*1024)
            if not block: break
            out.write(block); written+=len(block)
            if written-last >= 40*1024*1024:
                print(round(written/1024/1024),'MB downloaded',flush=True); last=written
    if sha256_file(partial) != ARCHIVE_SHA256: raise ValueError('Downloaded archive differs; partial retained. Do not change the expected checksum to force acceptance.')
    partial.replace(ARCHIVE)
with zipfile.ZipFile(ARCHIVE) as z:
    bad=z.testzip()
    if bad: raise ValueError('ZIP CRC failure: '+bad)
    names={}
    for item in z.infolist():
        if not item.is_dir(): names.setdefault(Path(item.filename).name,[]).append(item.filename)
    metadata_bytes=z.read(names['metadata.csv'][0]) if len(names.get('metadata.csv',[]))==1 else None
    if metadata_bytes is None: raise ValueError('Missing or ambiguous bundle metadata.')
    if hashlib.sha256(metadata_bytes).hexdigest() != audit['files']['milk10k.csv']['sha256']:
        raise ValueError('Bundle metadata differs from notebook 11.')
    source=pd.read_csv(io.BytesIO(metadata_bytes))
print('Official bundle checksum, CRC and audit metadata checks passed.')

## Fingerprint original and external reference images
Reads original test image bytes only to exclude duplicates; does not calculate test predictions or inspect test performance. Byte hashes, decoded RGB pixel hashes and a 64-bit difference hash are used. A difference-hash distance ≤4 is conservatively excluded as a **possible** near-duplicate, not proof of the same lesion. Rotation/crop variants and undocumented patient overlap may remain; patient independence is not established.

Both members of each MILK pair are checked. A duplicate dermoscopic image excludes its clinical partner, even when the clinical photo itself differs.

In [ ]:
def fingerprints(raw):
    with Image.open(io.BytesIO(raw)) as image:
        if image.width*image.height > 20_000_000: raise ValueError('Image exceeds 20 million pixels.')
        image=image.convert('RGB')
        pixel=hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
        small=np.asarray(image.convert('L').resize((9,8)),dtype=np.int16)
        bits=(small[:,1:]>small[:,:-1]).reshape(-1)
        dh=sum(int(bit)<<i for i,bit in enumerate(bits))
        width,height=image.size
    return {'image_hash':hashlib.sha256(raw).hexdigest(),'pixel_hash':pixel,'dhash':str(dh),'width':width,'height':height}

byte_seen={}; pixel_seen={}; dh_seen=[]
def remember(fp, identifier):
    byte_seen.setdefault(fp['image_hash'],identifier)
    pixel_seen.setdefault(fp['pixel_hash'],identifier)
    dh_seen.append((int(fp['dhash']),identifier))
def duplicate_match(fp):
    if fp['image_hash'] in byte_seen: return 'byte_duplicate',byte_seen[fp['image_hash']]
    if fp['pixel_hash'] in pixel_seen: return 'decoded_pixel_duplicate',pixel_seen[fp['pixel_hash']]
    d=int(fp['dhash'])
    for other,identifier in dh_seen:
        if (d^other).bit_count() <=4: return 'possible_near_duplicate',identifier
    return None,None

reference_audit=[]
for n,row in enumerate(references.to_dict('records'),1):
    path=Path(row['image_path'])
    if not path.is_file(): raise FileNotFoundError('Reference image missing; no incomplete screening allowed: '+str(path))
    fp=fingerprints(path.read_bytes())
    expected_hash=row.get('image_hash')
    if pd.notna(expected_hash) and fp['image_hash'] != expected_hash: raise ValueError('Reference bytes changed: '+str(path))
    identifier=str(row['reference_source'])+':'+str(row['image_id'])
    remember(fp,identifier)
    reference_audit.append({'reference':identifier,**fp})
    if n%500==0: print('Reference images screened:',n,flush=True)
pd.DataFrame(reference_audit).to_csv(OUTPUT_ROOT/'reference_fingerprints.csv',index=False)
print('Reference fingerprinting complete:',len(references))

## Screen clinical candidates and extract accepted photographs
Ambiguous members, unreadable files or missing pairs stop preparation. Duplicate exclusions are logged and may be reviewed; they are never silently restored. Cached clinical files must match the verified archive bytes.

In [ ]:
accepted=[]; excluded=[]; failures=[]
with zipfile.ZipFile(ARCHIVE) as z:
    for n,row in enumerate(candidates.sort_values('isic_id').to_dict('records'),1):
        try:
            pair=source.loc[source.lesion_id.eq(row['lesion_id'])].copy()
            if len(pair)!=2 or set(pair.image_type)!={'clinical: close-up','dermoscopic'}:
                raise ValueError('Expected exactly one clinical and one dermoscopic image per lesion.')
            images=[]
            for member in pair.to_dict('records'):
                filenames=names.get(member['isic_id']+'.jpg',[])
                if len(filenames)!=1: raise ValueError('Missing or ambiguous image: '+member['isic_id'])
                info=z.getinfo(filenames[0])
                if info.file_size>20_000_000: raise ValueError('Archive member exceeds byte limit.')
                raw=z.read(info)
                fp=fingerprints(raw)
                images.append((member,raw,fp))
            reason=match=matched_image=None
            for member,raw,fp in images:
                reason,match=duplicate_match(fp)
                if reason:
                    matched_image=member['isic_id'];break
            if reason:
                excluded.append({**row,'exclusion_reason':reason,'matched_reference':match,'screened_pair_member':matched_image})
            else:
                member,raw,fp=next(x for x in images if x[0]['image_type']=='clinical: close-up')
                target=IMAGE_ROOT/(row['isic_id']+'.jpg')
                if target.exists():
                    if sha256_file(target)!=fp['image_hash']: raise ValueError('Cached clinical image differs from verified bundle: '+str(target))
                else:
                    temporary=target.with_suffix('.jpg.part');temporary.write_bytes(raw);temporary.replace(target)
                accepted.append({**row,**fp,'image_path':str(target)})
            # Include all successfully screened MILK images, even exclusions, as references.
            # This conservatively prevents retaining another copy of an excluded lesion.
            for member,raw,fp in images: remember(fp,'MILK10k:'+member['isic_id'])
        except Exception as exc:
            failures.append({'isic_id':row['isic_id'],'error':str(exc)})
        if n%250==0: print('Candidate pairs screened:',n,flush=True)
pd.DataFrame(excluded,columns=list(candidates.columns)+['exclusion_reason','matched_reference','screened_pair_member']).to_csv(OUTPUT_ROOT/'excluded_duplicates.csv',index=False)
pd.DataFrame(failures,columns=['isic_id','error']).to_csv(OUTPUT_ROOT/'image_errors.csv',index=False)
if failures: raise RuntimeError(f'{len(failures)} image errors. Inspect image_errors.csv, resolve and rerun; no split manifests exported.')
clean=pd.DataFrame(accepted)
if clean.empty: raise RuntimeError('No accepted images.')
print('Accepted clinical photographs:',len(clean),'Excluded:',len(excluded),'Errors:',len(failures))
display(clean.class_label.value_counts())

## Freeze prospective MILK splits before model fitting
Create 70/15/15 stratified partitions with seed 42 using the screened clinical images. Split proportions are approximate due to rounding. These partitions are separate from the original dataset splits. Use only MILK train for fitting, MILK validation for development, and reserve MILK test for evaluation after candidate selection.

One clinical photograph per recorded lesion is enforced. No patient IDs are available here: absence of image overlap does not prove patient independence. Fitzpatrick training data remains training-only. Small tone subgroups cannot establish reliability.

In [ ]:
def assign_splits(frame, seed=42):
    frame=frame.sort_values('isic_id').reset_index(drop=True).copy()
    rng=np.random.default_rng(seed)
    frame['split']=''
    for label in CLASSES:
        idx=frame.index[frame.class_label.eq(label)].to_numpy()
        if len(idx)<20: raise ValueError('Too few accepted images for prospective class partitions: '+label)
        idx=rng.permutation(idx)
        nval=max(1,round(len(idx)*0.15)); ntest=max(1,round(len(idx)*0.15))
        frame.loc[idx[:nval],'split']='validation'
        frame.loc[idx[nval:nval+ntest],'split']='test'
        frame.loc[idx[nval+ntest:],'split']='train'
    return frame

prepared=assign_splits(clean,SEED)
prepared['image_id']=prepared.isic_id+'.jpg'
prepared['dataset']='MILK10k'
prepared['label_index']=prepared.class_label.map({label:i for i,label in enumerate(CLASSES)})
prepared['group_id']='MILK_'+prepared.lesion_id
prepared['patient_id']=pd.NA
prepared['fitzpatrick']=pd.NA
prepared['fitzpatrick_source']='Not available; MILK tone is a separate scale'
prepared['biopsy_confirmed']=True
prepared['training_ready']=prepared.split.eq('train')
assert prepared.image_hash.nunique()==len(prepared)
assert prepared.pixel_hash.nunique()==len(prepared)
assert prepared.groupby('group_id').split.nunique().max()==1
assert prepared.split.isin(['train','validation','test']).all()
output_hashes={}
for split in ['train','validation','test']:
    path=SPLIT_ROOT/f'{split}_metadata.csv'
    prepared.loc[prepared.split.eq(split)].to_csv(path,index=False)
    output_hashes[split]=sha256_file(path)
prepared.to_csv(OUTPUT_ROOT/'screened_clinical_manifest.csv',index=False)
tone_table=prepared.groupby(['split','class_label','skin_tone_class'],dropna=False).size().rename('image_count').reset_index()
tone_table.to_csv(OUTPUT_ROOT/'split_class_tone_counts.csv',index=False)
settings={'source_doi':'10.34970/648456','attribution':'MILK study team','licence':'CC-BY-NC',
          'archive_sha256':ARCHIVE_SHA256,'audit_report':str(AUDIT_ROOT),
          'candidate_manifest_sha256':sha256_file(candidate_path),
          'original_split_sha256':audit['original_split_sha256'],
          'external_training_manifest_sha256':sha256_file(EXTERNAL_MANIFEST),
          'split_manifest_sha256':output_hashes,'seed':SEED,'near_duplicate_dhash_distance':4,
          'paired_modalities_screened':True,'accepted':len(prepared),'excluded':len(excluded),
          'image_errors':len(failures),'patient_independence_established':False,
          'MILK_tone_is_Fitzpatrick':False,'test_predictions_performed':False}
(SPLIT_ROOT/'preparation_settings.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
print('MILK10k IMAGE PREPARATION COMPLETE')
display(pd.crosstab(prepared.split,prepared.class_label).reindex(['train','validation','test']))
display(tone_table)
print('Accepted:',len(prepared),'Excluded:',len(excluded),'Image errors:',len(failures))
print('Prepared split manifests:',SPLIT_ROOT)
print('Reports:',OUTPUT_ROOT)
print('No model training, test inference or app replacement performed.')

## Save and report
Save the executed notebook to Drive, and to GitHub as `notebooks/12_milk10k_image_preparation.ipynb`.
Commit message: `Prepare screened MILK10k clinical images and prospective splits`.

Share the final summary, class-by-split table, and prepared split folder. The training notebook must pin this folder and its checksums; do not regenerate splits while comparing experiments. MILK tone annotations remain separate from Fitzpatrick. Synthetic data, if added later, must be generated from training sources only and evaluated on real held-out images.